# SPEED-EXP-006 — VS13 Known-Speed Calibration / Tuning

Amaç: VS13 veri setindeki bilinen sabit hızlara sahip kısa videolar üzerinde mevcut tek kamera hız adayımızı test etmek.

Bu notebook **yeni neural speed modeli eğitmez**. Önce mevcut `YOLO + ByteTrack + bbox-geometry speed candidate` hattını çalıştırır, sonra kalibrasyon/parametre optimizasyonu yapar.

Varsayılan başlangıç paketleri:

- `RenaultCaptur.zip` — 66 km/h
- `KiaSportage.zip` — 72 km/h
- `VWPassat.zip` — 85 km/h

Çıktı yorumu:

- İyi sonuç: Mevcut hız adayımız bilinen hızlı videoda kalibrasyonla makul hata aralığına inebilir.
- Kötü sonuç: Hız modülü FTR için yine `relative/support evidence` kalır; mutlak km/s iddiası büyütülmez.

In [ ]:
# Cell 1 — Install dependencies
# Colab runtime: GPU recommended but not mandatory. L4/T4 is enough for this subset.

!pip -q install ultralytics pandas matplotlib tqdm opencv-python

In [ ]:
# Cell 2 — Imports and configuration
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import random
import shutil
import statistics
import subprocess
import time
import zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
from ultralytics import YOLO

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

EXPERIMENT_ID = 'SPEED-EXP-006-VS13-KNOWN-SPEED-CALIBRATION'
PROJECT_DRIVE_ROOT = Path('/content/drive/MyDrive/anomali-road-safety-ai')
LOCAL_WORK_ROOT = Path('/content/anomali-road-safety-ai-speed-exp-006')

MOUNT_DRIVE = True
USE_DRIVE_CACHE = True
FORCE_REDOWNLOAD = False
FORCE_REEXTRACT = False

# Start small. Increase after the first sanity run works.
MAX_VIDEOS_PER_VEHICLE = 6
FRAME_STRIDE = 1
TRACKER = 'bytetrack.yaml'
IMG_SIZE = 960
DETECT_CONF = 0.35
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

# If you copied a fine-tuned checkpoint to Drive, set it here.
# Otherwise the notebook uses Ultralytics yolo11n.pt from the package/cache.
DRIVE_MODEL_CANDIDATES = [
    PROJECT_DRIVE_ROOT / 'models/checkpoints/vehicle_detection/VD-EXP-002-GENERAL-YOLO11N-best.pt',
    PROJECT_DRIVE_ROOT / 'yolo11n.pt',
]
FALLBACK_MODEL = 'yolo11n.pt'

VS13_BASE = 'https://slobodan.ucg.ac.me/science/vs13/video+annotations'
VS13_PACKAGES = {
    'RenaultCaptur': {
        'url': f'{VS13_BASE}/RenaultCaptur.zip',
        'gt_speed_kmh': 66.0,
        'split': 'train',
    },
    'KiaSportage': {
        'url': f'{VS13_BASE}/KiaSportage.zip',
        'gt_speed_kmh': 72.0,
        'split': 'val',
    },
    'VWPassat': {
        'url': f'{VS13_BASE}/VWPassat.zip',
        'gt_speed_kmh': 85.0,
        'split': 'test',
    },
}
DESCRIPTION_URL = f'{VS13_BASE}/%20Description%20(video%2Bannotations).pdf'

# Parameter grid. Keep modest for first run; expand later.
PARAM_GRID = {
    'horizontal_fov_deg': [55.0, 60.0, 65.0, 70.0, 75.0],
    'vehicle_height_m': [1.45, 1.55, 1.65],
    'moving_average_window': [15, 25, 35],
    'max_segment_speed_kmh': [140.0, 180.0],
}

print('IN_COLAB:', IN_COLAB)
print('EXPERIMENT_ID:', EXPERIMENT_ID)

In [ ]:
# Cell 3 — Mount Drive and create folders
if IN_COLAB and MOUNT_DRIVE:
    drive.mount('/content/drive')

RAW_DIR = (PROJECT_DRIVE_ROOT if USE_DRIVE_CACHE else LOCAL_WORK_ROOT) / 'data/external/vs13/raw'
EXTRACT_DIR = LOCAL_WORK_ROOT / 'data/external/vs13/extracted_subset'
ARTIFACT_DIR = PROJECT_DRIVE_ROOT / 'models/benchmarks/artifacts/speed/SPEED-EXP-006-VS13-known-speed-calibration'
RUN_DIR = PROJECT_DRIVE_ROOT / 'runs/speed/SPEED-EXP-006-VS13-known-speed-calibration'
PLOT_DIR = RUN_DIR / 'plots'

for d in [RAW_DIR, EXTRACT_DIR, ARTIFACT_DIR, RUN_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('RAW_DIR:', RAW_DIR)
print('EXTRACT_DIR:', EXTRACT_DIR)
print('ARTIFACT_DIR:', ARTIFACT_DIR)
print('RUN_DIR:', RUN_DIR)

In [ ]:
# Cell 4 — Download VS13 packages to Drive/local cache

def download_file(url: str, target: Path, min_size_mb: float = 1.0, force: bool = False):
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() and target.stat().st_size > min_size_mb * 1024 * 1024 and not force:
        print('exists:', target, 'size MB:', round(target.stat().st_size / 1024 / 1024, 2))
        return target
    part = target.with_suffix(target.suffix + '.part')
    if part.exists():
        part.unlink()
    cmd = ['wget', '-O', str(part), '--tries=5', '--timeout=60', '--continue', url]
    print('+', ' '.join(cmd))
    subprocess.run(cmd, check=True)
    if part.stat().st_size < min_size_mb * 1024 * 1024:
        raise RuntimeError(f'Download too small: {part} size={part.stat().st_size}')
    part.rename(target)
    print('downloaded:', target, 'size MB:', round(target.stat().st_size / 1024 / 1024, 2))
    return target

# Description PDF is small; useful for citation/review.
download_file(DESCRIPTION_URL, RAW_DIR / 'Description_video_annotations.pdf', min_size_mb=0.05, force=False)

zip_paths = {}
for vehicle, meta in VS13_PACKAGES.items():
    target = RAW_DIR / f'{vehicle}.zip'
    zip_paths[vehicle] = download_file(meta['url'], target, min_size_mb=50.0, force=FORCE_REDOWNLOAD)

zip_paths

In [ ]:
# Cell 5 — Inspect zips and extract a small subset of MP4 + annotation files

def list_zip_preview(zip_path: Path, limit: int = 30):
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
    print('\nZIP:', zip_path.name, 'files:', len(names))
    for name in names[:limit]:
        print(' ', name)
    return names

for zp in zip_paths.values():
    list_zip_preview(zp, limit=12)


def extract_subset_for_vehicle(vehicle: str, zip_path: Path, max_videos: int):
    out_dir = EXTRACT_DIR / vehicle
    if out_dir.exists() and not FORCE_REEXTRACT:
        existing = sorted(out_dir.rglob('*.mp4'))
        if len(existing) >= max_videos:
            print(vehicle, 'already extracted videos:', len(existing))
            return existing[:max_videos]
    if FORCE_REEXTRACT and out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        mp4s = sorted([n for n in names if n.lower().endswith('.mp4')])
        selected_mp4s = mp4s[:max_videos]
        selected_stems = {Path(n).stem for n in selected_mp4s}
        selected = []
        for n in names:
            stem = Path(n).stem
            suffix = Path(n).suffix.lower()
            if n in selected_mp4s or (stem in selected_stems and suffix in {'.txt', '.csv', '.json'}):
                selected.append(n)
        print(vehicle, 'selected mp4:', len(selected_mp4s), 'selected files:', len(selected))
        for n in tqdm(selected, desc=f'extract {vehicle}'):
            zf.extract(n, out_dir)
    return sorted(out_dir.rglob('*.mp4'))[:max_videos]

video_paths = []
for vehicle, zp in zip_paths.items():
    video_paths.extend(extract_subset_for_vehicle(vehicle, zp, MAX_VIDEOS_PER_VEHICLE))

print('Extracted videos:', len(video_paths))
for p in video_paths[:20]:
    print(p)

In [ ]:
# Cell 6 — Build VS13 manifest

def infer_vehicle_from_path(path: Path):
    text = str(path).lower()
    for vehicle in VS13_PACKAGES:
        if vehicle.lower() in text:
            return vehicle
    return path.parts[-2]

records = []
for p in sorted(EXTRACT_DIR.rglob('*.mp4')):
    vehicle = infer_vehicle_from_path(p)
    meta = VS13_PACKAGES.get(vehicle)
    if not meta:
        continue
    records.append({
        'video_path': str(p),
        'video_name': p.name,
        'vehicle': vehicle,
        'gt_speed_kmh': float(meta['gt_speed_kmh']),
        'split': meta['split'],
    })

manifest_df = pd.DataFrame(records).sort_values(['split', 'vehicle', 'video_name']).reset_index(drop=True)
manifest_path = ARTIFACT_DIR / 'vs13_subset_manifest.csv'
manifest_df.to_csv(manifest_path, index=False)
print('manifest:', manifest_path, manifest_df.shape)
display(manifest_df.groupby(['split', 'vehicle', 'gt_speed_kmh']).size().reset_index(name='video_count'))
display(manifest_df.head(20))

In [ ]:
# Cell 7 — Resolve model checkpoint

def resolve_model_path():
    for p in DRIVE_MODEL_CANDIDATES:
        if p.exists():
            print('Using Drive model:', p)
            return str(p)
    print('Using fallback model:', FALLBACK_MODEL)
    return FALLBACK_MODEL

MODEL_PATH = resolve_model_path()
model = YOLO(MODEL_PATH)
print('model names:', model.names)

In [ ]:
# Cell 8 — Speed candidate functions

VEHICLE_CLASS_NAMES = {'car', 'motorcycle', 'bus', 'truck'}

def now_utc():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace('+00:00', 'Z')


def vehicle_class_ids(model):
    names = getattr(model, 'names', {}) or {}
    ids = [int(i) for i, name in names.items() if str(name) in VEHICLE_CLASS_NAMES]
    if not ids:
        # COCO fallback: car/motorcycle/bus/truck ids are usually 2/3/5/7.
        ids = [2, 3, 5, 7]
    return ids


def focal_lengths(width, height, horizontal_fov_deg):
    hfov = math.radians(horizontal_fov_deg)
    fx = width / (2.0 * math.tan(hfov / 2.0))
    vfov = 2.0 * math.atan(math.tan(hfov / 2.0) * (height / max(width, 1)))
    fy = height / (2.0 * math.tan(vfov / 2.0))
    return fx, fy


def moving_average(values, window):
    out = []
    for idx in range(len(values)):
        sample = [v for v in values[max(0, idx-window+1):idx+1] if v is not None and math.isfinite(v)]
        out.append(float(statistics.fmean(sample)) if sample else None)
    return out


def robust_mean(values):
    vals = np.array([v for v in values if v is not None and math.isfinite(v)], dtype=float)
    if len(vals) == 0:
        return None
    if len(vals) >= 8:
        lo, hi = np.percentile(vals, [10, 90])
        vals = vals[(vals >= lo) & (vals <= hi)]
    return float(np.mean(vals)) if len(vals) else None


def extract_main_track(video_path: Path, frame_stride=1):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f'Could not open video: {video_path}')
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    class_ids = vehicle_class_ids(model)
    model_names = getattr(model, 'names', {}) or {}

    obs_by_track = defaultdict(list)
    frame_id = 0
    start = time.perf_counter()
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame_id += 1
        if frame_stride > 1 and (frame_id - 1) % frame_stride != 0:
            continue
        results = model.track(
            frame,
            persist=True,
            tracker=TRACKER,
            classes=class_ids,
            conf=DETECT_CONF,
            imgsz=IMG_SIZE,
            device=DEVICE,
            verbose=False,
        )
        result = results[0]
        if result.boxes is None or result.boxes.id is None:
            continue
        ids = result.boxes.id.int().cpu().tolist()
        clss = result.boxes.cls.int().cpu().tolist()
        confs = result.boxes.conf.cpu().tolist()
        boxes = result.boxes.xyxy.cpu().tolist()
        for tid, cls_id, conf, box in zip(ids, clss, confs, boxes):
            x1, y1, x2, y2 = map(float, box)
            bw, bh = max(0, x2-x1), max(0, y2-y1)
            if bw <= 1 or bh <= 1:
                continue
            obs_by_track[int(tid)].append({
                'frame_id': frame_id,
                'time_s': frame_id / fps,
                'track_id': int(tid),
                'class_name': str(model_names.get(int(cls_id), cls_id)),
                'confidence': float(conf),
                'bbox_width_px': bw,
                'bbox_height_px': bh,
                'bbox_area_px': bw * bh,
                'bottom_center_x': (x1 + x2) / 2,
                'bottom_center_y': y2,
            })
    cap.release()

    tracks = []
    for tid, obs in obs_by_track.items():
        if len(obs) < 8:
            continue
        areas = [o['bbox_area_px'] for o in obs]
        confs = [o['confidence'] for o in obs]
        tracks.append({
            'track_id': tid,
            'observation_count': len(obs),
            'mean_area': float(np.mean(areas)),
            'median_area': float(np.median(areas)),
            'mean_confidence': float(np.mean(confs)),
            'obs': obs,
        })
    if not tracks:
        return {
            'status': 'failed',
            'failure_reason': 'no_track',
            'fps': fps,
            'frame_count': frame_count,
            'resolution': [width, height],
            'runtime_s': time.perf_counter() - start,
        }
    # Single-car pass-by videos: prefer long and large track.
    best = max(tracks, key=lambda t: (t['observation_count'], t['median_area'], t['mean_confidence']))
    return {
        'status': 'ok',
        'failure_reason': None,
        'fps': fps,
        'frame_count': frame_count,
        'resolution': [width, height],
        'runtime_s': time.perf_counter() - start,
        'track_count': len(tracks),
        'selected_track_id': best['track_id'],
        'selected_observation_count': best['observation_count'],
        'selected_mean_confidence': best['mean_confidence'],
        'observations': best['obs'],
    }


def compute_speed_candidate(track_result, params):
    if track_result.get('status') != 'ok':
        return {'status': 'failed', 'failure_reason': track_result.get('failure_reason')}
    obs = sorted(track_result['observations'], key=lambda x: x['frame_id'])
    width, height = track_result['resolution']
    fx, fy = focal_lengths(width, height, params['horizontal_fov_deg'])
    cx = width / 2.0
    vehicle_height_m = params['vehicle_height_m']
    rows = []
    prev = None
    raw_speeds = []
    for o in obs:
        bbox_h = max(o['bbox_height_px'], 1.0)
        z_m = fy * vehicle_height_m / bbox_h
        x_m = (o['bottom_center_x'] - cx) * z_m / fx
        row = dict(o)
        row['z_m'] = z_m
        row['x_m'] = x_m
        row['segment_speed_kmh_raw'] = None
        row['segment_valid'] = None
        row['segment_failure_reason'] = None
        if prev is not None:
            dt = max(1e-6, row['time_s'] - prev['time_s'])
            dx = row['x_m'] - prev['x_m']
            dz = row['z_m'] - prev['z_m']
            speed = math.sqrt(dx*dx + dz*dz) / dt * 3.6
            valid = True
            reason = None
            if not math.isfinite(speed) or speed > params['max_segment_speed_kmh']:
                valid = False
                reason = 'speed_outlier_gate'
            height_ratio = max(row['bbox_height_px'], prev['bbox_height_px']) / max(1.0, min(row['bbox_height_px'], prev['bbox_height_px']))
            if height_ratio > 1.25:
                valid = False
                reason = 'bbox_height_jump'
            row['segment_speed_kmh_raw'] = speed if valid else None
            row['segment_valid'] = valid
            row['segment_failure_reason'] = reason
            raw_speeds.append(row['segment_speed_kmh_raw'])
        rows.append(row)
        prev = row
    ma = moving_average(raw_speeds, params['moving_average_window'])
    # raw_speeds has len rows-1; align by skipping first row.
    for row, ma_value in zip(rows[1:], ma):
        row['segment_speed_kmh_moving_avg'] = ma_value
    if rows:
        rows[0]['segment_speed_kmh_moving_avg'] = None
    ma_values = [r.get('segment_speed_kmh_moving_avg') for r in rows]
    raw_values = [r.get('segment_speed_kmh_raw') for r in rows]
    estimate = robust_mean(ma_values)
    valid_count = sum(1 for v in raw_values if v is not None)
    speed_std = float(np.std([v for v in ma_values if v is not None])) if any(v is not None for v in ma_values) else None
    speed_mean = float(np.mean([v for v in ma_values if v is not None])) if any(v is not None for v in ma_values) else None
    speed_cv = (speed_std / speed_mean) if speed_mean and speed_mean > 0 else None
    confidence = 0.20
    confidence += 0.25 * min(len(obs) / 150.0, 1.0)
    confidence += 0.20 * min(valid_count / max(len(obs)-1, 1), 1.0)
    if estimate is not None:
        confidence += 0.15
    if speed_cv is not None:
        confidence += 0.15 * max(0.0, min(1.0, 1.0 - speed_cv / 1.2))
    confidence = round(min(confidence, 0.75), 4)
    return {
        'status': 'ok' if estimate is not None else 'failed',
        'failure_reason': None if estimate is not None else 'no_speed_estimate',
        'estimated_raw_kmh': estimate,
        'raw_mean_kmh': robust_mean(raw_values),
        'observation_count': len(obs),
        'valid_segment_count': valid_count,
        'speed_cv': speed_cv,
        'confidence': confidence,
        'timeseries': rows,
    }

In [ ]:
# Cell 9 — Run tracking and raw speed extraction once per video
# This is the slowest cell. It caches track/speed outputs in Drive.

TRACK_CACHE = ARTIFACT_DIR / 'vs13_track_speed_cache.json'

if TRACK_CACHE.exists():
    cache = json.loads(TRACK_CACHE.read_text())
    print('Loaded cache:', TRACK_CACHE, 'videos:', len(cache.get('videos', [])))
else:
    cache = {'experiment_id': EXPERIMENT_ID, 'created_at': now_utc(), 'videos': []}

cached_by_path = {v['video_path']: v for v in cache.get('videos', [])}

base_params = {
    'horizontal_fov_deg': 70.0,
    'vehicle_height_m': 1.55,
    'moving_average_window': 25,
    'max_segment_speed_kmh': 180.0,
}

new_rows = []
for rec in tqdm(manifest_df.to_dict('records'), desc='VS13 videos'):
    video_path = rec['video_path']
    if video_path in cached_by_path:
        new_rows.append(cached_by_path[video_path])
        continue
    track = extract_main_track(Path(video_path), frame_stride=FRAME_STRIDE)
    speed = compute_speed_candidate(track, base_params)
    row = dict(rec)
    row.update({
        'track_status': track.get('status'),
        'track_failure_reason': track.get('failure_reason'),
        'fps': track.get('fps'),
        'frame_count': track.get('frame_count'),
        'resolution': track.get('resolution'),
        'runtime_s': track.get('runtime_s'),
        'track_count': track.get('track_count'),
        'selected_track_id': track.get('selected_track_id'),
        'selected_observation_count': track.get('selected_observation_count'),
        'selected_mean_confidence': track.get('selected_mean_confidence'),
        'base_estimated_raw_kmh': speed.get('estimated_raw_kmh'),
        'base_confidence': speed.get('confidence'),
        'base_speed_cv': speed.get('speed_cv'),
        'track_result': track,
    })
    new_rows.append(row)
    cache['videos'] = new_rows
    TRACK_CACHE.write_text(json.dumps(cache, indent=2), encoding='utf-8')

cache['videos'] = new_rows
TRACK_CACHE.write_text(json.dumps(cache, indent=2), encoding='utf-8')
print('Wrote cache:', TRACK_CACHE)

base_df = pd.DataFrame([{k:v for k,v in row.items() if k not in {'track_result'}} for row in new_rows])
base_csv = ARTIFACT_DIR / 'vs13_base_speed_candidates.csv'
base_df.to_csv(base_csv, index=False)
print('base_csv:', base_csv)
display(base_df[['split','vehicle','video_name','gt_speed_kmh','track_status','selected_observation_count','base_estimated_raw_kmh','base_confidence','base_speed_cv']].head(50))

In [ ]:
# Cell 10 — Parameter grid search and global scale calibration

def iter_param_grid(grid):
    keys = list(grid.keys())
    def rec(idx, current):
        if idx == len(keys):
            yield dict(current)
            return
        key = keys[idx]
        for value in grid[key]:
            current[key] = value
            yield from rec(idx+1, current)
    yield from rec(0, {})


def mae(values):
    vals = [abs(v) for v in values if v is not None and math.isfinite(v)]
    return float(np.mean(vals)) if vals else None


def rmse(values):
    vals = [v for v in values if v is not None and math.isfinite(v)]
    return float(np.sqrt(np.mean(np.square(vals)))) if vals else None


def evaluate_params(rows, params):
    per_video = []
    for row in rows:
        track = row.get('track_result')
        speed = compute_speed_candidate(track, params) if track else {'status':'failed'}
        pred_raw = speed.get('estimated_raw_kmh')
        out = {k:v for k,v in row.items() if k != 'track_result'}
        out.update({
            'pred_raw_kmh': pred_raw,
            'candidate_confidence': speed.get('confidence'),
            'candidate_speed_cv': speed.get('speed_cv'),
            'candidate_status': speed.get('status'),
            'candidate_failure_reason': speed.get('failure_reason'),
        })
        per_video.append(out)
    df = pd.DataFrame(per_video)
    train_df = df[(df['split'] == 'train') & df['pred_raw_kmh'].notna() & (df['pred_raw_kmh'] > 0)]
    if len(train_df) == 0:
        return None, df
    ratios = train_df['gt_speed_kmh'].astype(float) / train_df['pred_raw_kmh'].astype(float)
    alpha = float(np.median(ratios))
    df['calibrated_speed_kmh'] = df['pred_raw_kmh'].astype(float) * alpha
    df['abs_error_kmh'] = (df['calibrated_speed_kmh'] - df['gt_speed_kmh'].astype(float)).abs()
    df['rel_error_pct'] = df['abs_error_kmh'] / df['gt_speed_kmh'].astype(float) * 100
    metrics = {
        **params,
        'global_scale_alpha': alpha,
    }
    for split in ['train','val','test']:
        sdf = df[df['split'] == split]
        errors = (sdf['calibrated_speed_kmh'] - sdf['gt_speed_kmh']).dropna().tolist() if len(sdf) else []
        metrics[f'{split}_count'] = int(len(sdf))
        metrics[f'{split}_mae_kmh'] = mae(errors)
        metrics[f'{split}_rmse_kmh'] = rmse(errors)
        metrics[f'{split}_mean_rel_error_pct'] = float(sdf['rel_error_pct'].dropna().mean()) if len(sdf) else None
    score = metrics.get('val_mae_kmh')
    if score is None:
        score = metrics.get('train_mae_kmh')
    metrics['selection_score'] = score
    return metrics, df

all_rows = new_rows
metrics_rows = []
best_metrics = None
best_df = None
for params in tqdm(list(iter_param_grid(PARAM_GRID)), desc='param grid'):
    metrics, df = evaluate_params(all_rows, params)
    if metrics is None:
        continue
    metrics_rows.append(metrics)
    if best_metrics is None or (metrics['selection_score'] is not None and metrics['selection_score'] < best_metrics['selection_score']):
        best_metrics = metrics
        best_df = df

metrics_df = pd.DataFrame(metrics_rows).sort_values(['selection_score','test_mae_kmh'], na_position='last').reset_index(drop=True)
metrics_csv = ARTIFACT_DIR / 'vs13_calibration_grid_metrics.csv'
metrics_df.to_csv(metrics_csv, index=False)
print('metrics_csv:', metrics_csv)
print('best_metrics:', json.dumps(best_metrics, indent=2))
display(metrics_df.head(20))

best_csv = ARTIFACT_DIR / 'vs13_best_calibrated_predictions.csv'
best_df.to_csv(best_csv, index=False)
print('best_csv:', best_csv)
display(best_df[['split','vehicle','video_name','gt_speed_kmh','pred_raw_kmh','calibrated_speed_kmh','abs_error_kmh','rel_error_pct','candidate_confidence','candidate_speed_cv']].head(80))

In [ ]:
# Cell 11 — Plots
plot_df = best_df.copy()

plt.figure(figsize=(8, 7))
for split, marker in [('train','o'), ('val','s'), ('test','^')]:
    sdf = plot_df[plot_df['split'] == split]
    plt.scatter(sdf['gt_speed_kmh'], sdf['calibrated_speed_kmh'], label=split, marker=marker, s=80)
lo = min(plot_df['gt_speed_kmh'].min(), plot_df['calibrated_speed_kmh'].min()) * 0.9
hi = max(plot_df['gt_speed_kmh'].max(), plot_df['calibrated_speed_kmh'].max()) * 1.1
plt.plot([lo, hi], [lo, hi], 'k--', linewidth=1)
plt.xlabel('Ground truth speed (km/h)')
plt.ylabel('Calibrated predicted speed (km/h)')
plt.title('VS13 calibrated speed sanity check')
plt.grid(True, alpha=0.3)
plt.legend()
scatter_path = PLOT_DIR / 'vs13_gt_vs_pred_scatter.png'
plt.savefig(scatter_path, dpi=160, bbox_inches='tight')
plt.show()
print(scatter_path)

plt.figure(figsize=(12, 5))
plot_df_sorted = plot_df.sort_values(['split','vehicle','video_name']).reset_index(drop=True)
labels = [f"{r.vehicle}\n{r.video_name[:12]}" for r in plot_df_sorted.itertuples()]
plt.bar(range(len(plot_df_sorted)), plot_df_sorted['abs_error_kmh'])
plt.xticks(range(len(plot_df_sorted)), labels, rotation=60, ha='right')
plt.ylabel('Absolute error (km/h)')
plt.title('VS13 calibrated speed absolute error')
plt.grid(axis='y', alpha=0.3)
error_path = PLOT_DIR / 'vs13_abs_error_by_video.png'
plt.savefig(error_path, dpi=160, bbox_inches='tight')
plt.show()
print(error_path)

# Confidence vs error
plt.figure(figsize=(8, 5))
plt.scatter(plot_df['candidate_confidence'], plot_df['abs_error_kmh'], s=80)
plt.xlabel('Candidate confidence')
plt.ylabel('Absolute error (km/h)')
plt.title('Confidence vs speed error')
plt.grid(True, alpha=0.3)
conf_path = PLOT_DIR / 'vs13_confidence_vs_error.png'
plt.savefig(conf_path, dpi=160, bbox_inches='tight')
plt.show()
print(conf_path)

In [ ]:
# Cell 12 — Write final summary report JSON + Markdown
summary = {
    'experiment_id': EXPERIMENT_ID,
    'created_at_utc': now_utc(),
    'purpose': 'Known-speed VS13 calibration sanity test for monocular speed candidates.',
    'model_path': MODEL_PATH,
    'dataset': {
        'name': 'VS13 audio-video vehicle speed estimation dataset',
        'source': 'https://slobodan.ucg.ac.me/science/vs13/',
        'packages': VS13_PACKAGES,
        'max_videos_per_vehicle': MAX_VIDEOS_PER_VEHICLE,
    },
    'best_metrics': best_metrics,
    'outputs': {
        'manifest_csv': str(manifest_path),
        'base_candidates_csv': str(base_csv),
        'grid_metrics_csv': str(metrics_csv),
        'best_predictions_csv': str(best_csv),
        'plots_dir': str(PLOT_DIR),
    },
    'limitations': [
        'VS13 speeds are maintained by cruise control; this is useful sanity ground truth but not LiDAR-grade enforcement truth.',
        'Camera viewpoint differs from the local demo videos, so calibration transfer is not guaranteed.',
        'This notebook tunes calibration parameters, not a new neural speed model.',
    ],
}
summary_json = ARTIFACT_DIR / 'speed_exp_006_vs13_known_speed_calibration_summary.json'
summary_json.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('summary_json:', summary_json)

report = ARTIFACT_DIR / 'speed_exp_006_vs13_known_speed_calibration_report.md'
lines = []
lines.append('# SPEED-EXP-006 VS13 Known-Speed Calibration')
lines.append('')
lines.append('This experiment tests the current monocular bbox-geometry speed candidate on VS13 videos with known constant speeds.')
lines.append('')
lines.append('## Best Parameters')
lines.append('')
for k, v in best_metrics.items():
    lines.append(f'- `{k}`: `{v}`')
lines.append('')
lines.append('## Outputs')
lines.append('')
for k, v in summary['outputs'].items():
    lines.append(f'- `{k}`: `{v}`')
lines.append('')
lines.append('## Interpretation')
lines.append('')
lines.append('If test MAE is acceptable, the speed module can be described as a dataset-calibrated approximate candidate. If not, keep it as relative/support evidence.')
lines.append('')
lines.append('## Limitations')
lines.append('')
for item in summary['limitations']:
    lines.append(f'- {item}')
report.write_text('\n'.join(lines), encoding='utf-8')
print('report:', report)

In [ ]:
# Cell 13 — Next-step guidance
print('Review these files in Drive:')
print('1)', ARTIFACT_DIR / 'vs13_best_calibrated_predictions.csv')
print('2)', ARTIFACT_DIR / 'vs13_calibration_grid_metrics.csv')
print('3)', ARTIFACT_DIR / 'speed_exp_006_vs13_known_speed_calibration_summary.json')
print('4)', PLOT_DIR)
print('\nDecision rule:')
print('- If TEST MAE is low enough for the report target, promote SPEED-EXP-006 as approximate dataset-calibrated speed candidate.')
print('- If TEST MAE is high or unstable, keep speed as relative/support evidence and do not claim absolute km/h.')